# Support Vector Machine (SVM)
**Repositori**: Machine Learning
**Topik**: Implementasi SVM dengan berbagai kernel pada dataset Iris
**Dataset**: Iris (built-in dari sklearn)
---
**Pendahuluan**: SVM adalah algoritma supervised learning yang mencari hyperplane optimal untuk memisahkan kelas. Notebook ini mengeksplorasi kernel Linear, RBF, dan Polynomial serta tuning hyperparameter.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from typing import Tuple, Optional
from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

logging.basicConfig(level=logging.INFO, format='%(message)s')
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 2. Load Dataset


In [ ]:
iris = datasets.load_iris()
X = iris.data
y = iris.target
logging.info(f'Shape X: {X.shape}')
logging.info(f'Jumlah kelas: {len(np.unique(y))}')


## 3. EDA


In [ ]:
df = pd.DataFrame(X, columns=iris.feature_names)
df['species'] = iris.target_names[y]
logging.info(f'Head:\n{df.head()}')
logging.info(f'Describe:\n{df.describe()}')
sns.pairplot(df, hue='species')
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 6))
sns.heatmap(df.iloc[:, :-1].corr(), annot=True, cmap='coolwarm')
plt.title('Korelasi Fitur Iris')
plt.tight_layout()
plt.show()


## 4. Data Preparation


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples')


## 5. SVM dengan Berbagai Kernel


In [ ]:
kernels = ['linear', 'rbf', 'poly']
results = {}
models = {}
for kernel in kernels:
    model = SVC(kernel=kernel, random_state=42)
    try:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)
        results[kernel] = acc
        models[kernel] = model
        logging.info(f'=== Kernel {kernel} ===')
        logging.info(f'Accuracy: {acc:.4f}')
        logging.info(f'\n{classification_report(y_test, y_pred, target_names=iris.target_names)}')
    except Exception as e:
        logging.error(f'Kernel {kernel} gagal: {e}')


## 6. Hyperparameter Tuning (GridSearchCV)


In [ ]:
C_VALUES = [0.1, 1, 10, 100]
GAMMA_VALUES = [0.01, 0.1, 1]
DEGREE_VALUES = [2, 3]
param_grid = [
    {'C': C_VALUES, 'kernel': ['linear']},
    {'C': C_VALUES, 'gamma': GAMMA_VALUES, 'kernel': ['rbf']},
    {'C': C_VALUES, 'gamma': GAMMA_VALUES, 'degree': DEGREE_VALUES, 'kernel': ['poly']},
]
grid = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy')
try:
    grid.fit(X_train_scaled, y_train)
    logging.info(f'Best params: {grid.best_params_}')
    logging.info(f'Best CV score: {grid.best_score_:.4f}')
    y_pred_best = grid.predict(X_test_scaled)
    logging.info(f'Test accuracy: {accuracy_score(y_test, y_pred_best):.4f}')
except Exception as e:
    logging.error(f'GridSearch gagal: {e}')


## 7. Visualisasi Decision Boundary (2D)


In [ ]:
MESH_STEP = 0.02


def plot_decision_boundary(
    X: np.ndarray,
    y: np.ndarray,
    model: SVC,
    title: str,
    xlabel: str = 'Fitur 1',
    ylabel: str = 'Fitur 2',
) -> None:
    """Plot decision boundary dari SVM classifier dalam 2D.

    Args:
        X: Data fitur 2D (n_samples, 2).
        y: Label target (n_samples,).
        model: Model SVM yang sudah dilatih.
        title: Judul plot.
        xlabel: Label sumbu X.
        ylabel: Label sumbu Y.
    """
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, MESH_STEP),
        np.arange(y_min, y_max, MESH_STEP),
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', cmap=plt.cm.RdYlBu)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(*scatter.legend_elements(), title='Kelas')


X_train_2d = X_train_scaled[:, :2]
svm_rbf = SVC(kernel='rbf', C=10, gamma=0.1, random_state=42)
svm_rbf.fit(X_train_2d, y_train)
plt.figure(figsize=(10, 6))
plot_decision_boundary(
    X_train_2d, y_train, svm_rbf,
    'SVM Decision Boundary (RBF - 2 Fitur Pertama)',
    xlabel=iris.feature_names[0],
    ylabel=iris.feature_names[1],
)
plt.tight_layout()
plt.show()


## 8. Kesimpulan
SVM dengan kernel RBF memberikan performa terbaik pada dataset Iris. Hyperparameter tuning dengan GridSearchCV meningkatkan akurasi. Kernel linear cocok untuk data yang terpisah secara linear, sementara RBF lebih fleksibel untuk pola non-linear.
